In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime
from datetime import datetime as dt

from pprint import pprint

import sys
sys.path.append("../FinRL")

import itertools
import warnings
warnings.filterwarnings("ignore")

# Preprocess fundamental data

In [2]:
fund_df = pd.read_csv("sp500_tickers_fundamental_quarterly_20250712.csv")

In [3]:
df_daily_price = pd.read_csv("sp500_tickers_daily_price_20250712.csv")

In [4]:
len(df_daily_price.tic.unique())

1069

In [5]:
len(fund_df.tic.unique())

500

## 1.1 Use Trade date instead of quarterly report date

In [6]:
fund_df['datadate'] = pd.to_datetime(fund_df['datadate'])
fund_df['datadate'] = fund_df['datadate'].dt.strftime('%Y%m%d').astype(int)

# use trade date instead of report quarterly date
times = list(fund_df['datadate']) # quarterly report date
for i in range(len(times)):
    quarter = (times[i] - int(times[i]/10000)*10000)
    if 1201 < quarter:
        times[i] = int(times[i]/10000 + 1)*10000 + 301
    if quarter <= 301:
        times[i] = int(times[i]/10000)*10000 + 301
    if 301 < quarter <= 601:
        times[i] = int(times[i]/10000)*10000 + 601
    if 601 < quarter <= 901:
        times[i] = int(times[i]/10000)*10000 + 901
    if 901 < quarter <= 1201:
        times[i] = int(times[i]/10000)*10000 + 1201
    time_tmp = times[i]
    year = int(time_tmp/10000)
    month = int(quarter/100)
    day = int(quarter - month*100)
#     if(time_tmp < 20171114):
#         while(dt.date(year, month, day).weekday() > 4):
#             time_tmp = time_tmp + 1
#             year = int(time_tmp/10000)
#             month = int((time_tmp - year*10000)/100)
#             day = int(time_tmp - year*10000 - month*100)
#         times[i] = time_tmp

In [7]:
times = pd.to_datetime(times,format='%Y%m%d')
fund_df['tradedate']=times

## 1.2 Calculate adjusted close price

In [8]:
fund_df['adj_close_q'] = fund_df.prccq/fund_df.adjex

### match tickers and gvkey for fundamental and price data

In [9]:
tic_to_gvkey = {}
df_daily_groups = list(df_daily_price.groupby('tic'))

In [10]:
for tic, df_ in df_daily_groups:
    tic_to_gvkey[tic] = df_.gvkey.iloc[0]

In [11]:
fund_df.shape

(52788, 681)

In [12]:
sum(np.isin(fund_df.tic, list(tic_to_gvkey.keys()))==False)

np.int64(0)

In [13]:
fund_df = fund_df[np.isin(fund_df.tic, list(tic_to_gvkey.keys()))]

In [14]:
fund_df.shape

(52788, 681)

In [15]:
sum(np.isin(fund_df.tic, list(tic_to_gvkey.keys()))==False)

np.int64(0)

In [16]:
len(fund_df.gvkey.unique())

500

In [17]:
fund_df['gvkey'] = [tic_to_gvkey[x] for x in fund_df['tic']]

In [18]:
len(fund_df.gvkey.unique())

500

In [19]:
fund_df['date'] = fund_df["tradedate"]
#fund_df.drop('tradedate', axis=1, inplace=True)

In [20]:
fund_df['date']=pd.to_datetime(fund_df['date'], format="%Y%m%d")
fund_df.drop_duplicates(["date","gvkey"], keep='last',inplace=True)

### Get next quarter's return

In [21]:
l_df = list(fund_df.groupby('gvkey'))
for tic,df in l_df:
    df.reset_index(inplace=True, drop=True)
    df.sort_values('date')
    # our goal is to predict next quarter's return
    df['y_return'] = np.log(df['adj_close_q'].shift(-1) / df['adj_close_q'])

In [22]:
fund_df = pd.concat([x[1] for x in l_df])

In [23]:
fund_df.shape

(52788, 683)

In [24]:
fund_df.head()

,costat,curcdq,datafmt,indfmt,consol,gvkey,datadate,conm,tic,cusip,...,dvpspq,dvpsxq,mkvaltq,prccq,prchq,prclq,tradedate,adj_close_q,date,y_return
0,A,USD,STD,INDL,C,1075,20210630,PINNACLE WEST CAPITAL CORP,PNW,723484101,...,0.83,0.83,9244.9045,81.97,88.540,80.61,2021-09-01,81.97,2021-09-01,-0.124700
1,A,USD,STD,INDL,C,1075,20210930,PINNACLE WEST CAPITAL CORP,PNW,723484101,...,0.83,0.83,8163.3658,72.36,86.870,71.40,2021-12-01,72.36,2021-12-01,-0.024765
2,A,USD,STD,INDL,C,1075,20211231,PINNACLE WEST CAPITAL CORP,PNW,723484101,...,0.85,0.85,7971.5169,70.59,74.400,62.78,2022-03-01,70.59,2022-03-01,0.101102
3,A,USD,STD,INDL,C,1075,20220331,PINNACLE WEST CAPITAL CORP,PNW,723484101,...,0.85,0.85,8825.0657,78.10,78.685,66.15,2022-06-01,78.10,2022-06-01,-0.065888
4,A,USD,STD,INDL,C,1075,20220630,PINNACLE WEST CAPITAL CORP,PNW,723484101,...,0.85,0.85,8265.2654,73.12,80.510,65.13,2022-09-01,73.12,2022-09-01,-0.125282


## 1.3 Calculate Financial Ratios

In [25]:
fund_df['pe'] = fund_df.prccq / fund_df.epspxq
fund_df['ps'] = fund_df.prccq / (fund_df.revtq/fund_df.cshoq)
fund_df['pb'] = fund_df.prccq / ((fund_df.atq-fund_df.ltq)/fund_df.cshoq)

In [26]:
items = [
    'date', # Date
    'gvkey',#gvkey unique identifier
    'tic', # Ticker
    'gsector',#gics sector 11
    'oiadpq', # Quarterly operating income
    'revtq', # Quartely revenue
    'niq', # Quartely net income
    'atq', # Total asset
    'teqq', # Shareholder's equity
    'epspiy', # EPS(Basic) incl. Extraordinary items
    'ceqq', # Common Equity
    'cshoq', # Common Shares Outstanding
    'dvpspq', # Dividends per share
    'actq', # Current assets
    'lctq', # Current liabilities
    'cheq', # Cash & Equivalent
    'rectq', # Recievalbles
    'cogsq', # Cost of  Goods Sold
    'invtq', # Inventories
    'apq',# Account payable
    'dlttq', # Long term debt
    'dlcq', # Debt in current liabilites
    'ltq', # Liabilities   
    'pe', #Price–earnings ratio
    'ps', #Price–sales ratio
    'pb', #Price-to-Book (P/B) Ratio
    'adj_close_q',#adjusted close price
    'y_return' #next quarter return
]

# Omit items that will not be used
fund_data = fund_df[items]


In [27]:
fund_data.head()

,date,gvkey,tic,gsector,oiadpq,revtq,niq,atq,teqq,epspiy,...,invtq,apq,dlttq,dlcq,ltq,pe,ps,pb,adj_close_q,y_return
0,2021-09-01,1075,PNW,55,278.386,1000.249,215.697,21061.578,5834.899,2.23,...,365.746,377.157,6863.091,783.373,15226.679,42.916230,9.242603,1.584416,81.97,-0.124700
1,2021-12-01,1075,PNW,55,429.373,1308.254,339.798,21536.424,6186.464,5.24,...,381.041,353.591,7241.993,405.078,15349.960,24.039867,6.239894,1.319553,72.36,-0.024765
2,2022-03-01,1075,PNW,55,50.017,798.857,27.584,22003.222,6021.460,5.48,...,367.167,393.083,7642.136,542.443,15981.762,294.125000,9.978653,1.323851,70.59,0.101102
3,2022-06-01,1075,PNW,55,54.492,783.531,16.956,22200.954,6050.136,0.15,...,379.059,343.255,7962.342,363.899,16150.818,520.666667,11.263199,1.458656,78.10,-0.065888
4,2022-09-01,1075,PNW,55,223.095,1061.669,164.312,22501.480,6021.523,1.60,...,401.482,436.308,7951.806,666.320,16479.957,50.427586,7.785162,1.372620,73.12,-0.125282


In [28]:
fund_data.shape

(52788, 28)

In [29]:
# Rename column names for the sake of readability
fund_data = fund_data.rename(columns={
    'oiadpq':'op_inc_q', # Quarterly operating income
    'revtq':'rev_q', # Quartely revenue
    'niq':'net_inc_q', # Quartely net income
    'atq':'tot_assets', # Assets
    'teqq':'sh_equity', # Shareholder's equity
    'epspiy':'eps_incl_ex', # EPS(Basic) incl. Extraordinary items
    'ceqq':'com_eq', # Common Equity
    'cshoq':'sh_outstanding', # Common Shares Outstanding
    'dvpspq':'div_per_sh', # Dividends per share
    'actq':'cur_assets', # Current assets
    'lctq':'cur_liabilities', # Current liabilities
    'cheq':'cash_eq', # Cash & Equivalent
    'rectq':'receivables', # Receivalbles
    'cogsq':'cogs_q', # Cost of  Goods Sold
    'invtq':'inventories', # Inventories
    'apq': 'payables',# Account payable
    'dlttq':'long_debt', # Long term debt
    'dlcq':'short_debt', # Debt in current liabilites
    'ltq':'tot_liabilities', # Liabilities   
})

In [30]:
fund_data.head()

,date,gvkey,tic,gsector,op_inc_q,rev_q,net_inc_q,tot_assets,sh_equity,eps_incl_ex,...,inventories,payables,long_debt,short_debt,tot_liabilities,pe,ps,pb,adj_close_q,y_return
0,2021-09-01,1075,PNW,55,278.386,1000.249,215.697,21061.578,5834.899,2.23,...,365.746,377.157,6863.091,783.373,15226.679,42.916230,9.242603,1.584416,81.97,-0.124700
1,2021-12-01,1075,PNW,55,429.373,1308.254,339.798,21536.424,6186.464,5.24,...,381.041,353.591,7241.993,405.078,15349.960,24.039867,6.239894,1.319553,72.36,-0.024765
2,2022-03-01,1075,PNW,55,50.017,798.857,27.584,22003.222,6021.460,5.48,...,367.167,393.083,7642.136,542.443,15981.762,294.125000,9.978653,1.323851,70.59,0.101102
3,2022-06-01,1075,PNW,55,54.492,783.531,16.956,22200.954,6050.136,0.15,...,379.059,343.255,7962.342,363.899,16150.818,520.666667,11.263199,1.458656,78.10,-0.065888
4,2022-09-01,1075,PNW,55,223.095,1061.669,164.312,22501.480,6021.523,1.60,...,401.482,436.308,7951.806,666.320,16479.957,50.427586,7.785162,1.372620,73.12,-0.125282


In [31]:
# set data type to series
date = fund_data['date'].to_frame('date').reset_index(drop=True)
tic = fund_data['tic'].to_frame('tic').reset_index(drop=True)
gvkey = fund_data['gvkey'].to_frame('gvkey').reset_index(drop=True)
adj_close_q = fund_data['adj_close_q'].to_frame('adj_close_q').reset_index(drop=True)
y_return = fund_data['y_return'].to_frame('y_return').reset_index(drop=True)
gsector = fund_data['gsector'].to_frame('gsector').reset_index(drop=True)
pe = fund_data['pe'].to_frame('pe').reset_index(drop=True)
ps = fund_data['ps'].to_frame('ps').reset_index(drop=True)
pb = fund_data['pb'].to_frame('pb').reset_index(drop=True)

In [32]:
# Calculate financial ratios

# Profitability ratios
# Operating Margin
OPM = pd.Series(np.empty(fund_data.shape[0],dtype=object),name='OPM')
for i in range(0, fund_data.shape[0]):
    if i-3 < 0:
        OPM[i] = np.nan
    elif fund_data.iloc[i,1] != fund_data.iloc[i-3,1]:
        OPM.iloc[i] = np.nan
    else:
        OPM.iloc[i] = np.sum(fund_data['op_inc_q'].iloc[i-3:i])/np.sum(fund_data['rev_q'].iloc[i-3:i])
OPM=pd.Series(OPM).to_frame().reset_index(drop=True)

# Net Profit Margin        
NPM = pd.Series(np.empty(fund_data.shape[0],dtype=object),name='NPM')
for i in range(0, fund_data.shape[0]):
    if i-3 < 0:
        NPM[i] = np.nan
    elif fund_data.iloc[i,1] != fund_data.iloc[i-3,1]:
        NPM.iloc[i] = np.nan
    else:
        NPM.iloc[i] = np.sum(fund_data['net_inc_q'].iloc[i-3:i])/np.sum(fund_data['rev_q'].iloc[i-3:i])
NPM=pd.Series(NPM).to_frame().reset_index(drop=True)

# Return On Assets
ROA = pd.Series(np.empty(fund_data.shape[0],dtype=object),name='ROA')
for i in range(0, fund_data.shape[0]):
    if i-3 < 0:
        ROA[i] = np.nan
    elif fund_data.iloc[i,1] != fund_data.iloc[i-3,1]:
        ROA.iloc[i] = np.nan
    else:
        ROA.iloc[i] = np.sum(fund_data['net_inc_q'].iloc[i-3:i])/fund_data['tot_assets'].iloc[i]
ROA=pd.Series(ROA).to_frame().reset_index(drop=True)

# Return on Equity
ROE = pd.Series(np.empty(fund_data.shape[0],dtype=object),name='ROE')
for i in range(0, fund_data.shape[0]):
    if i-3 < 0:
        ROE[i] = np.nan
    elif fund_data.iloc[i,1] != fund_data.iloc[i-3,1]:
        ROE.iloc[i] = np.nan
    else:
        ROE.iloc[i] = np.sum(fund_data['net_inc_q'].iloc[i-3:i])/fund_data['sh_equity'].iloc[i]        
ROE=pd.Series(ROE).to_frame().reset_index(drop=True)

# For calculating valuation ratios in the next subpart, calculate per share items in advance
# Earnings Per Share       
EPS = fund_data['eps_incl_ex'].to_frame('EPS').reset_index(drop=True)

# Book Per Share
BPS = (fund_data['com_eq']/fund_data['sh_outstanding']).to_frame('BPS').reset_index(drop=True) # Need to check units

#Dividend Per Share
DPS = fund_data['div_per_sh'].to_frame('DPS').reset_index(drop=True)

# Liquidity ratios
# Current ratio
cur_ratio = (fund_data['cur_assets']/fund_data['cur_liabilities']).to_frame('cur_ratio').reset_index(drop=True)

# Quick ratio
quick_ratio = ((fund_data['cash_eq'] + fund_data['receivables'] )/fund_data['cur_liabilities']).to_frame('quick_ratio').reset_index(drop=True)

# Cash ratio
cash_ratio = (fund_data['cash_eq']/fund_data['cur_liabilities']).to_frame('cash_ratio').reset_index(drop=True)


# Efficiency ratios
# Inventory turnover ratio
inv_turnover = pd.Series(np.empty(fund_data.shape[0],dtype=object),name='inv_turnover')
for i in range(0, fund_data.shape[0]):
    if i-3 < 0:
        inv_turnover[i] = np.nan
    elif fund_data.iloc[i,1] != fund_data.iloc[i-3,1]:
        inv_turnover.iloc[i] = np.nan
    else:
        inv_turnover.iloc[i] = np.sum(fund_data['cogs_q'].iloc[i-3:i])/fund_data['inventories'].iloc[i]
inv_turnover=pd.Series(inv_turnover).to_frame().reset_index(drop=True)

# Receivables turnover ratio       
acc_rec_turnover = pd.Series(np.empty(fund_data.shape[0],dtype=object),name='acc_rec_turnover')
for i in range(0, fund_data.shape[0]):
    if i-3 < 0:
        acc_rec_turnover[i] = np.nan
    elif fund_data.iloc[i,1] != fund_data.iloc[i-3,1]:
        acc_rec_turnover.iloc[i] = np.nan
    else:
        acc_rec_turnover.iloc[i] = np.sum(fund_data['rev_q'].iloc[i-3:i])/fund_data['receivables'].iloc[i]
acc_rec_turnover=pd.Series(acc_rec_turnover).to_frame().reset_index(drop=True)

# Payable turnover ratio
acc_pay_turnover = pd.Series(np.empty(fund_data.shape[0],dtype=object),name='acc_pay_turnover')
for i in range(0, fund_data.shape[0]):
    if i-3 < 0:
        acc_pay_turnover[i] = np.nan
    elif fund_data.iloc[i,1] != fund_data.iloc[i-3,1]:
        acc_pay_turnover.iloc[i] = np.nan
    else:
        acc_pay_turnover.iloc[i] = np.sum(fund_data['cogs_q'].iloc[i-3:i])/fund_data['payables'].iloc[i]
acc_pay_turnover=pd.Series(acc_pay_turnover).to_frame().reset_index(drop=True)

## Leverage financial ratios
# Debt ratio
debt_ratio = (fund_data['tot_liabilities']/fund_data['tot_assets']).to_frame('debt_ratio').reset_index(drop=True)

# Debt to Equity ratio
debt_to_equity = (fund_data['tot_liabilities']/fund_data['sh_equity']).to_frame('debt_to_equity').reset_index(drop=True)

In [33]:
# Create a dataframe that merges all the ratios
ratios = pd.concat([date,gvkey,tic,gsector,adj_close_q,y_return,OPM,NPM,ROA,ROE,EPS,BPS,DPS,
                    cur_ratio,quick_ratio,cash_ratio,inv_turnover,acc_rec_turnover,acc_pay_turnover,
                   debt_ratio,debt_to_equity,pe,ps,pb], axis=1).reset_index(drop=True)

In [34]:
ratios.shape

(52788, 24)

In [35]:
# Replace NAs infinite values with zero
final_ratios = ratios.copy()
final_ratios = final_ratios.fillna(0)
final_ratios = final_ratios.replace(np.inf,0)

In [36]:
final_ratios.to_csv('final_ratios_raw.csv')

In [37]:
final_ratios.shape

(52788, 24)

In [38]:
final_ratios.head()

,date,gvkey,tic,gsector,adj_close_q,y_return,OPM,NPM,ROA,ROE,...,quick_ratio,cash_ratio,inv_turnover,acc_rec_turnover,acc_pay_turnover,debt_ratio,debt_to_equity,pe,ps,pb
0,2021-09-01,1075,PNW,55,81.97,-0.124700,0.000000,0.000000,0.000000,0.000000,...,0.172250,0.006992,0.000000,0.000000,0.000000,0.722960,2.609587,42.916230,9.242603,1.584416
1,2021-12-01,1075,PNW,55,72.36,-0.024765,0.000000,0.000000,0.000000,0.000000,...,0.261048,0.015575,0.000000,0.000000,0.000000,0.712744,2.481217,24.039867,6.239894,1.319553
2,2022-03-01,1075,PNW,55,70.59,0.101102,0.000000,0.000000,0.000000,0.000000,...,0.214323,0.005674,0.000000,0.000000,0.000000,0.726337,2.654134,294.125000,9.978653,1.323851
3,2022-06-01,1075,PNW,55,78.10,-0.065888,0.243865,0.187644,0.026264,0.096375,...,0.196627,0.008748,4.897731,10.358108,5.408600,0.727483,2.669497,520.666667,11.263199,1.458656
4,2022-09-01,1075,PNW,55,73.12,-0.125282,0.184693,0.132959,0.017081,0.063827,...,0.210799,0.014433,4.572683,7.278629,4.207693,0.732394,2.736842,50.427586,7.785162,1.372620


In [39]:
features_column_financial=[ 'OPM', 'NPM', 'ROA', 'ROE', 'EPS', 'BPS', 'DPS', 'cur_ratio',
       'quick_ratio', 'cash_ratio', 'inv_turnover', 'acc_rec_turnover',
       'acc_pay_turnover', 'debt_ratio', 'debt_to_equity', 'pe', 'ps', 'pb']

In [40]:
def handle_nan(df,features_column_financial):
    ##handle nan, inf
    df=df.drop(list(df[df.adj_close_q==0].index)).reset_index(drop=True)
    df['y_return'] = pd.to_numeric(df['y_return'], errors='coerce')
    for col in features_column_financial:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['y_return'].replace([np.nan,np.inf,-np.inf], np.nan, inplace=True)
    df[features_column_financial].replace([np.nan, np.inf, -np.inf], np.nan, inplace=True)
    dropped_col = []
    for col in features_column_financial:
        if np.any(~np.isfinite(df[col])):
            df.drop(columns=[col], axis=1, inplace=True)
    df.dropna(axis=0, inplace=True)
    df=df.reset_index(drop=True)
    print("dropped_col: ",dropped_col)
    return df

In [41]:
final_ratios=handle_nan(final_ratios,features_column_financial)

dropped_col:  []


In [42]:
final_ratios.shape

(50996, 21)

In [43]:
#final_ratios[final_ratios.adj_close_q==0]

In [44]:
final_ratios.date=final_ratios.date.apply(lambda x: x.strftime('%Y-%m-%d'))

In [45]:
final_ratios.shape

(50996, 21)

In [46]:
final_ratios.head()

,date,gvkey,tic,gsector,adj_close_q,y_return,ROA,ROE,EPS,BPS,...,cur_ratio,quick_ratio,cash_ratio,acc_rec_turnover,acc_pay_turnover,debt_ratio,debt_to_equity,pe,ps,pb
0,2021-09-01,1075,PNW,55,81.97,-0.124700,0.000000,0.000000,2.23,50.695347,...,0.752833,0.172250,0.006992,0.000000,0.000000,0.722960,2.609587,42.916230,9.242603,1.584416
1,2021-12-01,1075,PNW,55,72.36,-0.024765,0.000000,0.000000,5.24,53.759068,...,1.036314,0.261048,0.015575,0.000000,0.000000,0.712744,2.481217,24.039867,6.239894,1.319553
2,2022-03-01,1075,PNW,55,70.59,0.101102,0.000000,0.000000,5.48,52.301044,...,0.882877,0.214323,0.005674,0.000000,0.000000,0.726337,2.654134,294.125000,9.978653,1.323851
3,2022-06-01,1075,PNW,55,78.10,-0.065888,0.026264,0.096375,0.15,52.484314,...,0.998662,0.196627,0.008748,10.358108,5.408600,0.727483,2.669497,520.666667,11.263199,1.458656
4,2022-09-01,1075,PNW,55,73.12,-0.125282,0.017081,0.063827,1.60,52.268540,...,0.889095,0.210799,0.014433,7.278629,4.207693,0.732394,2.736842,50.427586,7.785162,1.372620


In [47]:
#final_ratios=final_ratios[final_ratios.date<'2022-12-01'].reset_index(drop=True)

In [48]:
#final_ratios.shape

In [49]:
final_ratios.to_csv('final_ratios.csv', index=False)

## 1.4 Separate by sector

In [50]:
for sec, df in final_ratios.groupby('gsector'):
    df.to_excel(f"sector{int(sec)}.xlsx", index=False)